In [1]:
from pathlib import Path

carpeta = Path("../data")

for archivo in carpeta.rglob("*"):
    if archivo.is_file() and archivo.suffix.lower() in [".csv", ".parquet", ".xlsx", ".json"]:
        print(archivo)

..\data\processed\demanda_eda_2024.parquet
..\data\raw\openmeteo_madrid_2024.parquet
..\data\raw\redata_demanda_2024.parquet


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.dummy import DummyRegressor
import joblib

# Rutas relativas a la carpeta notebooks
carpeta_datos = Path("../data")
carpeta_raw = carpeta_datos / "raw"
carpeta_procesados = carpeta_datos / "processed"
carpeta_modelos = Path("../models")

carpeta_raw.mkdir(parents=True, exist_ok=True)
carpeta_procesados.mkdir(parents=True, exist_ok=True)
carpeta_modelos.mkdir(parents=True, exist_ok=True)

print("Carpetas preparadas")
print("Datos:", carpeta_datos.resolve())
print("Modelos:", carpeta_modelos.resolve())

Carpetas preparadas
Datos: C:\Users\jafcp\Desktop\prediccion-electricidad-git\prediccion-electricidad\data
Modelos: C:\Users\jafcp\Desktop\prediccion-electricidad-git\prediccion-electricidad\models


In [5]:
import os
from getpass import getpass

token = getpass("Introduce tu token de ESIOS: ")
os.environ["ESIOS_API_KEY"] = token

print("Token configurado para esta sesión.")


Introduce tu token de ESIOS:  ········


Token configurado para esta sesión.


In [8]:
from esios import ESIOSClient

cliente = ESIOSClient()
indicador_pvpc = cliente.indicators.get(1001)

precios = indicador_pvpc.historical(
    "2025-01-01",
    "2025-01-07",
    chunk_workers=1
)

print("Filas:", len(precios))
print("Columnas:", precios.columns.tolist())
display(precios.head())


Failed to write C:\Users\jafcp\.cache\esios\geos.json: [WinError 183] No se puede crear un archivo que ya existe: 'C:\\Users\\jafcp\\.cache\\esios\\tmp4lr0eg72.json' -> 'C:\\Users\\jafcp\\.cache\\esios\\geos.json'
Failed to write C:\Users\jafcp\.cache\esios\indicators\1001\meta.json: [WinError 183] No se puede crear un archivo que ya existe: 'C:\\Users\\jafcp\\.cache\\esios\\indicators\\1001\\tmpm7n033_y.json' -> 'C:\\Users\\jafcp\\.cache\\esios\\indicators\\1001\\meta.json'
Failed to write C:\Users\jafcp\.cache\esios\geos.json: [WinError 183] No se puede crear un archivo que ya existe: 'C:\\Users\\jafcp\\.cache\\esios\\tmpx5zp52jp.json' -> 'C:\\Users\\jafcp\\.cache\\esios\\geos.json'
Failed to write C:\Users\jafcp\.cache\esios\geos.json: [WinError 183] No se puede crear un archivo que ya existe: 'C:\\Users\\jafcp\\.cache\\esios\\tmpjiae2_os.json' -> 'C:\\Users\\jafcp\\.cache\\esios\\geos.json'
Failed to write C:\Users\jafcp\.cache\esios\indicators\1001\meta.json: [WinError 183] No se 

Filas: 168
Columnas: ['Península', 'Canarias', 'Baleares', 'Ceuta', 'Melilla']


,Península,Canarias,Baleares,Ceuta,Melilla
datetime,,,,,
2025-01-01 00:00:00+01:00,182.79,182.79,182.79,182.79,182.79
2025-01-01 01:00:00+01:00,183.19,183.19,183.19,183.19,183.19
2025-01-01 02:00:00+01:00,188.74,188.74,188.74,188.74,188.74
2025-01-01 03:00:00+01:00,187.77,187.77,187.77,187.77,187.77
2025-01-01 04:00:00+01:00,180.19,180.19,180.19,180.19,180.19


In [9]:
from pathlib import Path

ruta_precios = Path("../data/raw/precios_pvpc_2025.parquet")
ruta_precios.parent.mkdir(parents=True, exist_ok=True)

precios.to_parquet(ruta_precios)

print("Precios guardados en:", ruta_precios.resolve())
print("Dimensiones:", precios.shape)

Precios guardados en: C:\Users\jafcp\Desktop\prediccion-electricidad-git\prediccion-electricidad\data\raw\precios_pvpc_2025.parquet
Dimensiones: (168, 5)


In [10]:
print("Columnas:", precios.columns.tolist())
print("Fechas:", precios.index.min(), "→", precios.index.max())
print("Valores nulos:\n", precios.isna().sum())
display(precios.head(10))

Columnas: ['Península', 'Canarias', 'Baleares', 'Ceuta', 'Melilla']
Fechas: 2025-01-01 00:00:00+01:00 → 2025-01-07 23:00:00+01:00
Valores nulos:
 Península    0
Canarias     0
Baleares     0
Ceuta        0
Melilla      0
dtype: int64


,Península,Canarias,Baleares,Ceuta,Melilla
datetime,,,,,
2025-01-01 00:00:00+01:00,182.79,182.79,182.79,182.79,182.79
2025-01-01 01:00:00+01:00,183.19,183.19,183.19,183.19,183.19
2025-01-01 02:00:00+01:00,188.74,188.74,188.74,188.74,188.74
2025-01-01 03:00:00+01:00,187.77,187.77,187.77,187.77,187.77
2025-01-01 04:00:00+01:00,180.19,180.19,180.19,180.19,180.19
2025-01-01 05:00:00+01:00,172.30,172.30,172.30,172.30,172.30
2025-01-01 06:00:00+01:00,171.28,171.28,171.28,171.28,171.28
2025-01-01 07:00:00+01:00,167.90,167.90,167.90,167.90,167.90
2025-01-01 08:00:00+01:00,165.98,165.98,165.98,165.98,165.98


In [11]:
print("Tipo de objeto:", type(precios))
print("Dimensiones:", precios.shape)
print("Columnas:", precios.columns.tolist())
print("Índice:", precios.index.name)
print("Fechas:", precios.index.min(), "→", precios.index.max())

display(precios.head())
display(precios.describe())

Tipo de objeto: <class 'pandas.DataFrame'>
Dimensiones: (168, 5)
Columnas: ['Península', 'Canarias', 'Baleares', 'Ceuta', 'Melilla']
Índice: datetime
Fechas: 2025-01-01 00:00:00+01:00 → 2025-01-07 23:00:00+01:00


,Península,Canarias,Baleares,Ceuta,Melilla
datetime,,,,,
2025-01-01 00:00:00+01:00,182.79,182.79,182.79,182.79,182.79
2025-01-01 01:00:00+01:00,183.19,183.19,183.19,183.19,183.19
2025-01-01 02:00:00+01:00,188.74,188.74,188.74,188.74,188.74
2025-01-01 03:00:00+01:00,187.77,187.77,187.77,187.77,187.77
2025-01-01 04:00:00+01:00,180.19,180.19,180.19,180.19,180.19


,Península,Canarias,Baleares,Ceuta,Melilla
count,168.000000,168.000000,168.000000,168.000000,168.000000
mean,152.760357,152.760357,152.760357,152.759524,152.759524
std,60.852161,60.852161,60.852161,60.148637,60.148637
min,29.120000,29.120000,29.120000,29.120000,29.120000
25%,107.737500,107.737500,107.737500,107.737500,107.737500
50%,151.905000,151.905000,151.905000,154.255000,154.255000
75%,188.012500,188.012500,188.012500,189.220000,189.220000
max,423.150000,423.150000,423.150000,423.150000,423.150000


In [12]:
# Nos quedamos con el precio de la Península
datos_precio = precios[["Península"]].copy()

# Convertimos el índice en una columna
datos_precio = datos_precio.reset_index()

# Renombramos las columnas
datos_precio.columns = ["datetime", "precio"]

# Ordenamos cronológicamente
datos_precio = datos_precio.sort_values("datetime").reset_index(drop=True)

# Creamos variables temporales
datos_precio["hora"] = datos_precio["datetime"].dt.hour
datos_precio["dia_semana"] = datos_precio["datetime"].dt.dayofweek
datos_precio["mes"] = datos_precio["datetime"].dt.month
datos_precio["dia_mes"] = datos_precio["datetime"].dt.day

print("Dimensiones:", datos_precio.shape)
display(datos_precio.head())

Dimensiones: (168, 6)


,datetime,precio,hora,dia_semana,mes,dia_mes
0,2025-01-01 00:00:00+01:00,182.79,0,2,1,1
1,2025-01-01 01:00:00+01:00,183.19,1,2,1,1
2,2025-01-01 02:00:00+01:00,188.74,2,2,1,1
3,2025-01-01 03:00:00+01:00,187.77,3,2,1,1
4,2025-01-01 04:00:00+01:00,180.19,4,2,1,1


In [13]:
# Variables que utilizará el modelo
variables = ["hora", "dia_semana", "mes", "dia_mes"]

X = datos_precio[variables]
y = datos_precio["precio"]

# Separación cronológica: primeras horas para entrenar,
# últimas horas para evaluar
punto_corte = int(len(datos_precio) * 0.8)

X_train = X.iloc[:punto_corte]
X_test = X.iloc[punto_corte:]

y_train = y.iloc[:punto_corte]
y_test = y.iloc[punto_corte:]

# Entrenamiento
modelo_precio = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    min_samples_leaf=2,
    n_jobs=-1
)

modelo_precio.fit(X_train, y_train)

# Predicciones
predicciones = modelo_precio.predict(X_test)

# Evaluación
mae = mean_absolute_error(y_test, predicciones)
rmse = np.sqrt(mean_squared_error(y_test, predicciones))

print(f"Filas de entrenamiento: {len(X_train)}")
print(f"Filas de prueba: {len(X_test)}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

Filas de entrenamiento: 134
Filas de prueba: 34
MAE: 74.64
RMSE: 87.47


In [14]:
resultado = datos_precio.iloc[punto_corte:][
    ["datetime", "precio"]
].copy()

resultado["precio_predicho"] = predicciones
resultado["error_absoluto"] = (
    resultado["precio"] - resultado["precio_predicho"]
).abs()

display(resultado.head(15))

,datetime,precio,precio_predicho,error_absoluto
134,2025-01-06 14:00:00+01:00,51.55,76.579489,25.029489
135,2025-01-06 15:00:00+01:00,52.40,77.290407,24.890407
136,2025-01-06 16:00:00+01:00,92.05,80.825995,11.224005
137,2025-01-06 17:00:00+01:00,125.12,82.923671,42.196329
138,2025-01-06 18:00:00+01:00,145.40,83.293680,62.106320
139,2025-01-06 19:00:00+01:00,165.64,83.237305,82.402695
140,2025-01-06 20:00:00+01:00,172.99,83.302236,89.687764
141,2025-01-06 21:00:00+01:00,179.23,83.283248,95.946752
142,2025-01-06 22:00:00+01:00,166.00,82.563248,83.436752
143,2025-01-06 23:00:00+01:00,148.48,82.563248,65.916752


In [15]:
ruta_modelo = carpeta_modelos / "modelo_precio_v01.joblib"

joblib.dump(modelo_precio, ruta_modelo)

print("Modelo guardado en:", ruta_modelo.resolve())

Modelo guardado en: C:\Users\jafcp\Desktop\prediccion-electricidad-git\prediccion-electricidad\models\modelo_precio_v01.joblib
